# Final application verification
Use synthetic responses only. Run against the published application after deployment. Browser observations are recorded separately using the written test cases. This notebook measures HTTP round trips and a small three-client concurrent scenario; it does not establish maximum capacity or guarantee that any request is a cold start.

In [1]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import csv, hashlib, json, math, re, shutil, statistics, subprocess, threading, time
from datetime import datetime, timezone
import requests

try:
    from google.colab import drive
except ImportError:
    ROOT=Path.cwd().resolve()
    if ROOT.name=='notebooks': ROOT=ROOT.parent
else:
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/Final Research - C D Jayasinghe')
OUT=ROOT/('evaluation/final' if (ROOT/'.git').exists() else 'outputs/final_verification')
OUT.mkdir(parents=True,exist_ok=True)
APP_URL='https://help-seeking-intention-dss.vercel.app'
features=[f'D{i:02}' for i in range(1,21)]
synthetic=dict.fromkeys(features,1)
def write_json(name,value):
    (OUT/name).write_text(json.dumps(value,indent=2,ensure_ascii=False),encoding='utf-8')
def write_csv(name,rows):
    if not rows: raise ValueError('No observations to save.')
    with (OUT/name).open('w',encoding='utf-8',newline='') as stream:
        writer=csv.DictWriter(stream,fieldnames=list(rows[0])); writer.writeheader();writer.writerows(rows)
def git_commit():
    if not (ROOT/'.git').exists(): return 'not supplied in this environment'
    return subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
session=requests.Session()
session.headers.update({'User-Agent':'Pathways-DSS-research-verification/1.0'})
started=datetime.now(timezone.utc).isoformat()
records=[]

## 1. Hosted functionality and response headers
Expected scores for synthetic Rarely responses: total 20/80 and five category sums of 4/16. Explicitly skipped D01: incomplete total, observed subtotal 19, possible total 19–23. Invalid inputs must be rejected without reflecting unnecessary submitted data.

In [2]:
responses={}
for path in ['/', '/api/health','/api/questionnaire','/api/support']:
    r=session.get(APP_URL+path,timeout=30)
    responses[path]=r
    records.append({'case':'GET '+path,'expected_status':'200','actual_status':r.status_code,
                    'passed':r.status_code==200,'details':'HTTPS response received'})
catalog=responses['/api/questionnaire'].json()
root_html=responses['/'].text
assets=re.findall(r'(?:src|href)="(/assets/[^" ]+)"',root_html)
source_commit=git_commit()
provenance={'started_utc':started,'tested_source_commit':source_commit,'app_url':APP_URL,
            'deployed_assets':assets,'request_data':'Synthetic questionnaire fixtures only',
            'browser_engines_available':['In-app browser only; see browser matrix'],
            'cold_start_confirmed':False}
write_json('verification_provenance.json',provenance)

def valid_result(data,partial=False):
    assert len(data['probabilities'])==3 and all(0<=p<=1 for p in data['probabilities'])
    assert math.isclose(sum(data['probabilities']),1,abs_tol=1e-8)
    assert len(data['contributions'])==20
    assert math.isclose(data['reference_probability']+sum(c['value'] for c in data['contributions']),
                        data['probabilities'][data['category']],abs_tol=1e-8)
    if partial:
        assert data['dvq']['score'] is None and data['dvq']['observed_subtotal']==19
        assert data['dvq']['possible_range']==[19,23]
        assert data['dvq']['domains'][0]['score'] is None
    else:
        assert data['dvq']['score']==20 and all(d['score']==4 for d in data['dvq']['domains'])

payload={'eligible':True,'responses':synthetic}
for case,body,partial in [('complete scoring',payload,False),
    ('partial scoring',{'eligible':True,'responses':{**synthetic,'D01':None}},True)]:
    r=session.post(APP_URL+'/api/predict',json=body,timeout=30)
    if not partial: responses['/api/predict']=r
    passed=r.status_code==200
    if passed:
        try: valid_result(r.json(),partial)
        except (AssertionError,KeyError,ValueError): passed=False
    records.append({'case':case,'expected_status':'200 and correct scores/probabilities/explanation',
                    'actual_status':r.status_code,'passed':passed,'details':'Synthetic complete or skipped-response fixture'})

invalid=[('eligibility false',{'eligible':False,'responses':synthetic}),
 ('extra personal field',{**payload,'name':'SYNTHETIC-REJECT-MARKER'}),
 ('missing responses',{'eligible':True,'responses':{}}),
 ('target leakage field',{'eligible':True,'responses':{**synthetic,'H3':7}}),
 ('all skipped',{'eligible':True,'responses':dict.fromkeys(features,None)}),
 ('array payload',[])]
for value in [-1,5,1.5,True,'1']:
    invalid.append((f'invalid response {value!r}',{'eligible':True,'responses':{**synthetic,'D01':value}}))
for case,body in invalid:
    r=session.post(APP_URL+'/api/predict',json=body,timeout=30)
    records.append({'case':case,'expected_status':'422','actual_status':r.status_code,
                    'passed':r.status_code==422 and 'SYNTHETIC-REJECT-MARKER' not in r.text,
                    'details':'Invalid request rejected without echoing marker'})
for path,method in [('/api/predict','GET'),('/api/health','POST'),('/api/questionnaire','POST'),('/api/support','POST')]:
    r=session.request(method,APP_URL+path,timeout=30)
    records.append({'case':method+' '+path,'expected_status':'405','actual_status':r.status_code,
                    'passed':r.status_code==405,'details':'Unsupported method rejected'})
r=session.post(APP_URL+'/api/predict',data='{broken-json',headers={'Content-Type':'application/json'},timeout=30)
records.append({'case':'malformed JSON','expected_status':'400 or 422','actual_status':r.status_code,
                'passed':r.status_code in [400,422],'details':'Malformed JSON rejected'})
r=session.post(APP_URL+'/api/predict',data='SYNTHETIC-PRIVATE-MARKER',headers={'Content-Type':'text/plain'},timeout=30)
records.append({'case':'malformed text privacy','expected_status':'422 and fixed error without submitted text',
                'actual_status':r.status_code,'passed':r.status_code==422 and 'SYNTHETIC' not in r.text,
                'details':'Regression for parser-error reflection'})
for private_path in ['/.env','/.git/config','/outputs/evaluation_private/sus_scoring.csv','/evaluation/user_expert_data_audit.json']:
    r=session.get(APP_URL+private_path,timeout=30)
    inaccessible=r.status_code in [403,404] or (r.status_code==200 and r.content==responses['/'].content)
    records.append({'case':'Private source inaccessible '+private_path,'expected_status':'403/404 or public HTML shell only',
                    'actual_status':r.status_code,'passed':inaccessible,'details':'No private file content exposed'})
write_csv('hosted_api_cases.csv',records)

header_checks=[]
for path,r in responses.items():
    checks={
        'HTTPS':r.url.startswith('https://'),
        'No response caching':'no-store' in r.headers.get('Cache-Control','').lower(),
        'MIME sniffing prevented':r.headers.get('X-Content-Type-Options','').lower()=='nosniff',
        'Referrer suppressed':r.headers.get('Referrer-Policy','').lower()=='no-referrer',
        'Framing denied':r.headers.get('X-Frame-Options','').upper()=='DENY',
        'HSTS present':'max-age=' in r.headers.get('Strict-Transport-Security',''),
        'Scripts restricted to own origin':"script-src 'self'" in r.headers.get('Content-Security-Policy',''),
        'No wildcard CORS on API responses':r.headers.get('Access-Control-Allow-Origin')!='*' if path.startswith('/api/') else True,
        'No application cookie set':'Set-Cookie' not in r.headers
    }
    for name,passed in checks.items():
        header_checks.append({'path':path,'check':name,'passed':passed,
                              'scope':'Observed HTTP response; public HTML wildcard CORS is permitted because it contains no questionnaire results; not a penetration test'})
write_csv('security_header_checks.csv',header_checks)
write_json('deployed_response_headers.json',{path:dict(r.headers) for path,r in responses.items()})
print(f'Hosted API checks: {sum(r["passed"] for r in records)}/{len(records)}')
print(f'Header checks: {sum(r["passed"] for r in header_checks)}/{len(header_checks)}')
assert all(r['passed'] for r in records), 'Resolve hosted API failures.'
assert all(r['passed'] for r in header_checks), 'Inspect deployed header failures.'

Hosted API checks: 27/27
Header checks: 45/45


## 2. Hosted response times and modest concurrency
Engineering targets declared before this run: no failed or incorrect requests and p95 HTTP response time at most 3000 ms for the sampled sequential and three-client scenarios. These are project targets, not clinical or universal performance standards. Record the first measured performance request (after functional checks) separately; serverless instance state is not observable, so it must not be labelled a confirmed cold start.

In [3]:
local_session=threading.local()
def measured_request(index,phase):
    if not hasattr(local_session,'session'):
        local_session.session=requests.Session()
    tick=time.perf_counter()
    status=None; error=''; passed=False
    try:
        r=local_session.session.post(APP_URL+'/api/predict',json=payload,timeout=30)
        status=r.status_code
        if status!=200: raise ValueError(f'HTTP {status}')
        valid_result(r.json())
        passed=True
    except (requests.RequestException,AssertionError,KeyError,ValueError) as exc:
        error=type(exc).__name__
    return {'phase':phase,'request':index,'elapsed_ms':round((time.perf_counter()-tick)*1000,3),
            'status':status,'correct':passed,'error':error}

timings=[measured_request(1,'first_measured_after_functional_checks')]
timings.extend(measured_request(i+1,'sequential') for i in range(20))
concurrent_start=time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as pool:
    timings.extend(pool.map(lambda i:measured_request(i,'concurrent_3'),range(1,13)))
concurrent_seconds=time.perf_counter()-concurrent_start

def quantile(values,p):
    ordered=sorted(values);position=(len(ordered)-1)*p;lower=math.floor(position)
    return ordered[lower]+(ordered[math.ceil(position)]-ordered[lower])*(position-lower)
summaries=[]
for phase in ['first_measured_after_functional_checks','sequential','concurrent_3']:
    observed=[r for r in timings if r['phase']==phase]
    values=[r['elapsed_ms'] for r in observed]
    p95=quantile(values,.95)
    summaries.append({'phase':phase,'n':len(values),'failed_or_incorrect_n':sum(not r['correct'] for r in observed),
                      'mean_ms':round(statistics.mean(values),3),'p50_ms':round(quantile(values,.5),3),
                      'p95_ms':round(p95,3),'minimum_ms':min(values),'maximum_ms':max(values),
                      'p95_target_ms':3000 if phase!='first_measured_after_functional_checks' else None,
                      'target_met':all(r['correct'] for r in observed) and p95<=3000 if phase!='first_measured_after_functional_checks' else None,
                      'scope':'Client HTTP round trip including response read, JSON parsing and correctness checks'})
write_csv('hosted_latency_requests.csv',timings)
write_csv('hosted_latency_summary.csv',summaries)
write_json('hosted_load_scope.json',{'max_concurrent_clients':3,'concurrent_request_n':12,
    'concurrent_wall_seconds':round(concurrent_seconds,3),
    'concurrent_observed_requests_per_second':round(12/concurrent_seconds,3),
    'cold_start_confirmed':False,'maximum_capacity_tested':False,
    'limitation':'No platform instance lifecycle control. First measured performance request follows the functional checks and is not a verified cold start. Small sample and one client location.'})
for row in summaries: print(row)

{'phase': 'first_measured_after_functional_checks', 'n': 1, 'failed_or_incorrect_n': 0, 'mean_ms': 507.976, 'p50_ms': 507.976, 'p95_ms': 507.976, 'minimum_ms': 507.976, 'maximum_ms': 507.976, 'p95_target_ms': None, 'target_met': None, 'scope': 'Client HTTP round trip including response read, JSON parsing and correctness checks'}
{'phase': 'sequential', 'n': 20, 'failed_or_incorrect_n': 0, 'mean_ms': 359.614, 'p50_ms': 357.98, 'p95_ms': 377.657, 'minimum_ms': 343.059, 'maximum_ms': 387.337, 'p95_target_ms': 3000, 'target_met': True, 'scope': 'Client HTTP round trip including response read, JSON parsing and correctness checks'}
{'phase': 'concurrent_3', 'n': 12, 'failed_or_incorrect_n': 0, 'mean_ms': 409.991, 'p50_ms': 354.337, 'p95_ms': 598.725, 'minimum_ms': 328.249, 'maximum_ms': 635.215, 'p95_target_ms': 3000, 'target_met': True, 'scope': 'Client HTTP round trip including response read, JSON parsing and correctness checks'}


## 3. Dependency, source and build checks
Run npm audits and existing application checks when a source checkout is available. Dependency audit results reflect the registry response for this lockfile and execution; zero findings do not establish absence of all vulnerabilities.

In [4]:
if (ROOT/'package.json').exists():
    npm=shutil.which('npm.cmd') or shutil.which('npm')
    if not npm: raise RuntimeError('Node/npm required for source checks.')
    dependency_rows=[]
    for name,folder in [('root',ROOT),('frontend',ROOT/'deployment/frontend')]:
        result=subprocess.run([npm,'audit','--json'],cwd=folder,capture_output=True,text=True)
        report=json.loads(result.stdout)
        write_json(f'dependency_audit_{name}.json',report)
        if 'error' in report: raise RuntimeError('Dependency audit unavailable; inspect saved report.')
        dependency_rows.append({'scope':name,**report['metadata']['vulnerabilities']})
    write_csv('dependency_audit_summary.csv',dependency_rows)
    assert all(r['total']==0 for r in dependency_rows), 'Triage the saved dependency findings before declaring the audit passed.'
    test_result=subprocess.run(['node','--test','--test-reporter=tap','tests/model.test.mjs','tests/dss.test.mjs'],cwd=ROOT,capture_output=True,text=True)
    (OUT/'application_tests.tap').write_text(test_result.stdout+test_result.stderr,encoding='utf-8')
    build_result=subprocess.run([npm,'run','build'],cwd=ROOT,capture_output=True,text=True)
    (OUT/'production_build.txt').write_text(build_result.stdout+build_result.stderr,encoding='utf-8')
    assert test_result.returncode==0, 'Application tests failed.'
    assert build_result.returncode==0, 'Production build failed.'
    built_assets={p.name for p in (ROOT/'deployment/frontend/dist/assets').iterdir()}
    assert all(Path(url).name in built_assets for url in assets), 'Hosted assets differ from this source build.'
    # Report only file/line references, never questionnaire records or secret values.
    patterns={
        'browser_persistence':r'localStorage|sessionStorage|indexedDB|document\.cookie',
        'application_logging':r'console\.(log|info|warn|error)|process\.stdout',
        'server_data_writes':r'writeFile|appendFile|createWriteStream',
        'external_transmission':r'fetch\(|axios|https?://',
        'dangerous_HTML':r'dangerouslySetInnerHTML|innerHTML\s*='
    }
    findings=[]
    source_files=[*list((ROOT/'api').glob('*.mjs')),*list((ROOT/'lib').glob('*.mjs')),
                  *list((ROOT/'deployment/frontend/src').glob('*.tsx')),
                  *list((ROOT/'deployment/frontend/src').glob('*.mjs'))]
    for file in source_files:
        for line_number,line in enumerate(file.read_text(encoding='utf-8').splitlines(),1):
            for check,pattern in patterns.items():
                if re.search(pattern,line):
                    findings.append({'check':check,'file':file.relative_to(ROOT).as_posix(),'line':line_number,
                                     'review':'Manual source inspection required; URL literals may be support links'})
    write_csv('security_source_references.csv',findings)
    print('Application tests and production build passed; hosted asset filenames match the source build.')
else:
    print('Source checkout not present. Dependency/source/build checks are not executed in this environment.')

Application tests and production build passed; hosted asset filenames match the source build.


## 4. Consolidation
Keep browser, accessibility and compatibility observations with their real execution scope. Native screen-reader, other browser engines, physical mobile devices and confirmed cold starts require separate evidence. Recruitment dates, signed-letter details and academic qualifications must come from the researcher or source records.

In [5]:
summary={
    'tested_source_commit':source_commit,'deployed_assets':assets,
    'api_checks':len(records),'api_passed':sum(r['passed'] for r in records),
    'header_checks':len(header_checks),'header_passed':sum(r['passed'] for r in header_checks),
    'performance_requests':len(timings),'performance_failures':sum(not r['correct'] for r in timings),
    'sequential_and_concurrent_targets_met':all(r['target_met'] for r in summaries if r['phase']!='first_measured_after_functional_checks'),
    'cold_start_confirmed':False,
    'completed_utc':datetime.now(timezone.utc).isoformat()
}
write_json('technical_verification_summary.json',summary)
print(json.dumps(summary,indent=2))

{
  "tested_source_commit": "d674a40b464b64c58a40218924f7850f5721b795",
  "deployed_assets": [
    "/assets/index-CRL-x_B_.js",
    "/assets/index-Bsc0MsTL.css"
  ],
  "api_checks": 27,
  "api_passed": 27,
  "header_checks": 45,
  "header_passed": 45,
  "performance_requests": 33,
  "performance_failures": 0,
  "sequential_and_concurrent_targets_met": true,
  "cold_start_confirmed": false,
  "completed_utc": "2026-10-24T04:56:22.726505+00:00"
}


## 5. Consolidated browser evidence and thesis tables
Use only the recorded observations. Environment-dependent checks stay explicitly unexecuted.

In [6]:
import platform,sys
write_json('verification_environment.json',{
    'local_python':sys.version.split()[0],'local_node':subprocess.check_output(['node','--version'],text=True).strip(),
    'os':platform.system()+' '+platform.release(),
    'deployed_node_major':'22.x; package engines and Vercel project setting',
    'ci_node_major':'22.x; .github/workflows/ci.yml',
    'client_scope':'One Windows execution client; browser engine not independently identified',
    'performance_request_count':33,'maximum_concurrent_clients':3})

def read_csv(name):
    with (OUT/name).open(encoding='utf-8-sig',newline='') as stream:
        return list(csv.DictReader(stream))

browser_path=OUT/'browser_regression.json'
if not browser_path.is_file():
    print('Browser observations unavailable. Transfer the actual recorded browser evidence before consolidation.')
else:
    browser=json.loads(browser_path.read_text())
    accessibility=json.loads((OUT/'accessibility_dom_audit.json').read_text())
    technical=json.loads((OUT/'technical_verification_summary.json').read_text())
    keyboard=json.loads((OUT/'keyboard_observations.json').read_text())
    assert {f'FT-{i:02}' for i in range(1,19)}.issubset({r['test_id'] for r in browser})
    assert len({r['test_id'] for r in browser})==len(browser)
    assert all(r['status']=='PASS' for r in browser)
    write_csv('browser_regression.csv',browser)

    def luminance(channels):
        values=[v/255 for v in channels]
        linear=[v/12.92 if v<=0.04045 else ((v+0.055)/1.055)**2.4 for v in values]
        return sum(v*w for v,w in zip(linear,[0.2126,0.7152,0.0722]))
    outline_rgb=[181,119,16]
    background_rgb=[241,247,250]
    light,dark=sorted([luminance(outline_rgb),luminance(background_rgb)],reverse=True)
    focus_ratio=(light+0.05)/(dark+0.05)
    accessibility_rows=[]
    for view in accessibility:
        for check,passed,actual in [
            ('Document language',view['lang']=='en',view['lang']),
            ('Document title',bool(view['title']),view['title']),
            ('One primary heading',view['h1']==1,str(view['h1'])),
            ('Visible control names',not view['unnamed_controls'],f"{len(view['unnamed_controls'])} unnamed"),
            ('Sampled visible text contrast',not view['contrast_failures'],f"{view['contrast_checked']} text elements; {len(view['contrast_failures'])} flagged"),
            ('Radio groups have legends',all(r['group'] for r in view['radio_groups']),f"{len(view['radio_groups'])} radio controls inspected")]:
            accessibility_rows.append({'view':view['view'],'check':check,'status':'PASS' if passed else 'FAIL','actual':actual,
                                       'scope':'Rendered DOM checks; not full WCAG conformance or screen-reader usability'})
    accessibility_rows.extend([
        {'view':'keyboard','check':'Space toggles eligibility','status':'PASS' if keyboard['checkbox']['checked'] else 'FAIL',
         'actual':'Real Space key selected the checkbox','scope':'In-app browser'},
        {'view':'keyboard','check':'Visible keyboard focus','status':'PASS' if keyboard['checkbox']['focusVisible'] and focus_ratio>=3 else 'FAIL',
         'actual':f'Visible outline; measured colour ratio {focus_ratio:.3f}:1 against sampled label background',
         'scope':'One observed checkbox focus state; CSS also supplies focus-visible rules for other controls'},
        {'view':'keyboard','check':'Arrow navigation in radio group','status':'PASS' if keyboard['radio_arrow_right_selected_value']=='1' else 'FAIL',
         'actual':'ArrowRight moved from Never to Rarely','scope':'In-app browser'},
        {'view':'native screen reader','check':'Usability with native screen reader','status':'NOT EXECUTED',
         'actual':'No native screen-reader control surface available','scope':'Accessibility tree inspection is not a substitute'}])
    write_csv('accessibility_checks.csv',accessibility_rows)

    references=read_csv('security_source_references.csv')
    reviews=[]
    for row in references:
        reviews.append({**row,'review':(
            'Same-origin API fetch; static endpoint paths; answers are sent only in the prediction request body.'
            if row['file'].endswith('main.tsx') and row['line']=='10' else
            'Reviewed literal support/evidence URL or explicit leave-page link; no questionnaire data appended to URLs.')})
    write_csv('security_source_references.csv',reviews)
    write_json('security_review_scope.json',{
        'application_source_reviewed':['api/*.mjs','lib/*.mjs','deployment/frontend/src/*.tsx','deployment/frontend/src/*.mjs'],
        'dependency_audits':'Saved npm audit reports; zero known vulnerabilities reported for both lockfiles.',
        'application_logging_references_found':any(r['check']=='application_logging' for r in references),
        'application_browser_persistence_found':any(r['check']=='browser_persistence' for r in references),
        'application_source_write_references_found':any(r['check']=='server_data_writes' for r in references),
        'unsafe_HTML_insertion_found':any(r['check']=='dangerous_HTML' for r in references),
        'data_transport':'Browser sends assessment responses to the same-origin prediction API over HTTPS. University and plan preferences stay in page state.',
        'download':'An explicit optional plan download is the intentional client file output; sensitive scores excluded by default.',
        'provider_logging':'Application source contains no answer logging. Vercel infrastructure metadata retention was not independently audited.',
        'excluded':['Formal penetration test','Provider-internal storage and logging audit','Continuous availability monitoring'],
        'public_HTML_CORS':'Wildcard CORS on the public HTML shell is not evidence of access to private response data; API responses are checked separately.'})

    defects=[
        {'id':'A11Y-01','finding':'Questionnaire progress had no accessible name','fix':'Added Questionnaire completion aria-label','verification':'No unnamed visible controls in seven inspected states','status':'FIXED','commit':'94b0ec0'},
        {'id':'A11Y-02','finding':'Six initial small-text contrast failures below 4.5:1','fix':'Darkened inactive step numbers and service source text','verification':'Zero text contrast failures in the sampled final states','status':'FIXED','commit':'94b0ec0'},
        {'id':'A11Y-03','finding':'Light-background keyboard focus outline had insufficient contrast','fix':'Darkened focus outline to #b57710','verification':f'Sampled outline contrast {focus_ratio:.3f}:1; real keyboard events checked','status':'FIXED','commit':'73c196c'},
        {'id':'SEC-01','finding':'Malformed JSON parser errors reflected a submitted text prefix','fix':'Return a fixed validation message; add automated regression','verification':'Local regression and hosted malformed-text case pass without echoing SYNTHETIC','status':'FIXED','commit':'d674a40'}]
    write_csv('resolved_defects.csv',defects)

    context=json.loads((OUT/'evaluation_context.json').read_text())
    timeline=json.loads((OUT/'evaluation_deployment_timeline.json').read_text())
    context['evaluated_app']['production_revision_during_period_inferred']=timeline['production_before_submissions']['commit']
    context['evaluated_app']['version_evidence']='evaluation_deployment_timeline.json'
    context['evaluated_app']['note']='Vercel production history indicates revision998e7bf during the observed evaluation period; no per-participant build or browser-cache identifier was captured. This is retrospective inference.'
    write_json('evaluation_context.json',context)

    status=[
        {'task':'Final functional regression','status':'COMPLETE within available environment',
         'result':f'All 18 original written cases passed; {len(browser)} total browser observations passed, including additional checks.',
         'evidence':'browser_regression.csv; plan_download_check.json; application_tests.tap; hosted_api_cases.csv',
         'remaining_or_limit':'One isolated local model-failure fixture; other cases hosted. Broader browser compatibility is listed separately.'},
        {'task':'Accessibility testing','status':'PARTIAL',
         'result':'Seven DOM page states checked with no unnamed controls or sampled text-contrast failures; keyboard Space/arrow/focus checks passed.',
         'evidence':'accessibility_checks.csv; accessibility_dom_audit.json; keyboard_observations.json',
         'remaining_or_limit':'Native screen-reader usability and a comprehensive WCAG audit are not executed.'},
        {'task':'Cross-browser testing','status':'PARTIAL',
         'result':'In-app browser desktop and 390x844 viewport checks passed.',
         'evidence':'compatibility_matrix.csv; dashboard_complete_desktop.jpg; dashboard_mobile.jpg',
         'remaining_or_limit':'Chrome, Edge, Firefox and physical mobile-device checks need those actual environments; none is connected.'},
        {'task':'Hosted performance testing','status':'PARTIAL: measured scenarios complete',
         'result':f"33 correct responses; sequential p95 {read_csv('hosted_latency_summary.csv')[1]['p95_ms']} ms; three-client p95 {read_csv('hosted_latency_summary.csv')[2]['p95_ms']} ms.",
         'evidence':'hosted_latency_requests.csv; hosted_latency_summary.csv; hosted_load_scope.json',
         'remaining_or_limit':'No confirmed cold-start observation or maximum-capacity/stress test. First performance sample follows functional requests.'},
        {'task':'Security checks','status':'COMPLETE within stated review scope',
         'result':f"14 automated tests; {technical['api_passed']}/{technical['api_checks']} hosted API/private-path checks; {technical['header_passed']}/{technical['header_checks']} header checks; zero known dependency vulnerabilities.",
         'evidence':'dependency_audit_summary.csv; security_header_checks.csv; security_review_scope.json; resolved_defects.csv',
         'remaining_or_limit':'A scoped technical review, not a formal penetration test or provider logging audit.'},
        {'task':'Evaluation context','status':'RECORDED with explicit evidence limits',
         'result':'Shared-link volunteer recruitment; Google Forms submissions on 23 October 2026; same app URL; domain roles provided; production revision inferred from timeline.',
         'evidence':'evaluation_context.json; submission_period_observations.json; evaluation_deployment_timeline.json',
         'remaining_or_limit':'Signed letters are held by researcher but not inspected; degree qualifications and per-participant build identifiers are not independently verified.'},
        {'task':'Final evidence consolidation','status':'COMPLETE for available evidence',
         'result':'Written cases, actual observations, unit/API results, security review, screenshots, latency chart and scope limitations consolidated.',
         'evidence':'evidence_manifest.json; final_verification_status.csv; hosted_latency.png',
         'remaining_or_limit':'Do not describe unexecuted browser, native screen-reader or confirmed cold-start checks as completed.'}]
    write_csv('final_verification_status.csv',status)

    # Keep the original written procedures and replace their old result references.
    written_path=ROOT/'evaluation/functional_test_cases.csv'
    if written_path.is_file():
        with written_path.open(encoding='utf-8-sig',newline='') as stream:
            reader=csv.DictReader(stream); fields=reader.fieldnames; written=list(reader)
        observations={r['test_id']:r for r in browser}
        extra_procedures={
            'FT-04b':('eligibility_after_answers','Answer all five first-page items without confirming eligibility.','Continue remains disabled until eligibility is confirmed.'),
            'FT-19':('back_navigation','Complete pages one and two, including an explicit skip; select Back.','The first-page answers and explicit skip are retained.'),
            'FT-20':('refresh_privacy','Complete a synthetic assessment, then reload the page.','No questionnaire responses or eligibility remain selected.'),
            'FT-21':('leave_page_link','Inspect the Leave this page destination and referrer attribute.','Destination is Google and referrer policy is no-referrer.')}
        existing_ids={r['test_id'] for r in written}
        for test_id,(scenario,steps,expected) in extra_procedures.items():
            if test_id not in existing_ids:
                written.append({'test_id':test_id,'scenario':scenario,'steps_and_test_data':steps,
                    'expected_result':expected,'recorded_actual_result':'','status':'','evidence':'','execution_scope':''})
        for row in written:
            obs=observations[row['test_id']]
            row['recorded_actual_result']=obs['actual_result'];row['status']=obs['status']
            row['evidence']='evaluation/final/browser_regression.csv; evaluation/final/technical_verification_summary.json'
            row['execution_scope']=obs['environment']+'; final frontend asset hashes matched across commits73c196c and d674a40; API regression verified at d674a40.'
        with written_path.open('w',encoding='utf-8',newline='') as stream:
            writer=csv.DictWriter(stream,fieldnames=fields);writer.writeheader();writer.writerows(written)

    # The original non-functional table remains a concise thesis index.
    nonfunctional_path=ROOT/'evaluation/nonfunctional_test_cases.csv'
    if nonfunctional_path.is_file():
        with nonfunctional_path.open(encoding='utf-8-sig',newline='') as stream:
            reader=csv.DictReader(stream); fields=reader.fieldnames; rows=list(reader)
        mapping={'NFT-06':4,'NFT-07':1,'NFT-08':3,'NFT-09':2}
        for row in rows:
            if row['test_id'] not in mapping: continue
            target=status[mapping[row['test_id']]]
            row['status']=target['status'];row['recorded_result']=target['result']
            row['evidence']='evaluation/final/final_verification_status.csv; '+target['evidence']
            row['limitation']=target['remaining_or_limit']
        specific={
            'NFT-02':('PASS within tested viewports','Desktop and 390x844 assessment/dashboard workflows fit without horizontal overflow; scores and controls readable.','evaluation/final/browser_regression.csv; evaluation/final/dashboard_mobile.jpg','One available browser and viewport emulation only; physical mobile devices and other engines not tested.'),
            'NFT-03':('PASS within checked behaviors','Clear and reload remove answers; real plan download excludes scores by default and raw responses; malformed parser errors no longer reflect submitted text.','evaluation/final/plan_download_check.json; evaluation/final/browser_regression.csv; evaluation/final/security_review_scope.json','Application behavior and source reviewed; provider-internal logging and retention not independently audited.'),
            'NFT-04':('PASS within injected failure','With deliberately unavailable MODEL_DIR, assessment reports unavailability while support and the Medical care filter remain usable.','evaluation/final/browser_regression.csv; evaluation/final/model_failure_support.jpg','Isolated local failure fixture only; production was not deliberately disrupted.')}
        for row in rows:
            if row['test_id'] in specific:
                row['status'],row['recorded_result'],row['evidence'],row['limitation']=specific[row['test_id']]
        with nonfunctional_path.open('w',encoding='utf-8',newline='') as stream:
            writer=csv.DictWriter(stream,fieldnames=fields);writer.writeheader();writer.writerows(rows)


    # Refresh the familiar evidence filenames while retaining actual environment labels.
    case_names={r['test_id']:r['scenario'] for r in written} if written_path.is_file() else {}
    legacy_rows=[{'case':case_names.get(r['test_id'],r['test_id']),'passed':int(r['status']=='PASS'),
                  'notes':r['actual_result'],'environment':r['environment']} for r in browser]
    hosted_rows=[r for r in legacy_rows if r['environment'].startswith('Published')]
    for name,values in [('dss_browser_acceptance.csv',legacy_rows),('dss_hosted_browser_acceptance.csv',hosted_rows),('hosted_acceptance.csv',hosted_rows)]:
        with (ROOT/'evaluation'/name).open('w',encoding='utf-8',newline='') as stream:
            writer=csv.DictWriter(stream,fieldnames=list(values[0]));writer.writeheader();writer.writerows(values)
    for name in ['app_tests.txt','dss_tests.txt']:
        shutil.copyfile(OUT/'application_tests.tap',ROOT/'evaluation'/name)
    hosting_path=ROOT/'evaluation/dss_hosting_check.json'
    if hosting_path.is_file():
        hosting=json.loads(hosting_path.read_text());hosting['tested_source_commit']=technical['tested_source_commit']
        hosting['hosted_api_cases_passed']=technical['api_passed'];hosting['hosted_header_checks_passed']=technical['header_passed']
        hosting['browser_scope']='Final frontend assets checked in the in-app browser; isolated model-failure case tested locally.'
        hosting_path.write_text(json.dumps(hosting,indent=2),encoding='utf-8')

    latest_images={
        'assessment_desktop.jpg':['app_desktop.jpg','hosted_desktop.jpg'],
        'assessment_mobile.jpg':['app_mobile.jpg'],
        'dashboard_complete_desktop.jpg':['dss_dashboard_desktop.jpg'],
        'dashboard_mobile.jpg':['dss_dashboard_mobile.jpg','dss_hosted_mobile.jpg'],
        'support_desktop.jpg':['dss_support_services.jpg'],
        'model_failure_support.jpg':['dss_model_unavailable.jpg']}
    for source,targets in latest_images.items():
        for target in targets: shutil.copyfile(OUT/source,ROOT/'evaluation'/target)
    shutil.copyfile(OUT/'ci_check.json',ROOT/'evaluation/ci_check.json')
    manifest={'final_application_commit':technical['tested_source_commit'],
              'frontend_assets':technical['deployed_assets'],
              'browser_case_n':len(browser),'original_browser_case_n':18,
              'current_core_flow_repeated_after_final_API_fix':True,
              'browser_scope':'Original scenarios run against final frontend assets; parser-only backend fix then passed hosted API regression and a repeated complete hosted browser workflow.',
              'private_data_included':False,
              'excluded_evidence':['Native screen-reader sessions','Chrome/Edge/Firefox executions','Physical mobile test','Confirmed cold-start measurement'],
              'files':[]}
    for file in sorted(OUT.iterdir()):
        if file.is_file() and file.name!='evidence_manifest.json':
            manifest['files'].append({'file':file.name,'bytes':file.stat().st_size,
                                      'sha256':hashlib.sha256(file.read_bytes()).hexdigest()})
    write_json('evidence_manifest.json',manifest)
    print('Consolidated 18 original written functional cases, additional browser checks and explicit pending environment-dependent tests.')

Consolidated 18 original written functional cases, additional browser checks and explicit pending environment-dependent tests.


In [7]:
if (OUT/'hosted_latency_summary.csv').is_file():
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    with (OUT/'hosted_latency_summary.csv').open(encoding='utf-8',newline='') as stream:
        chart_rows=[r for r in csv.DictReader(stream) if r['phase'] in ['sequential','concurrent_3']]
    fig,ax=plt.subplots(figsize=(7.6,4.5))
    positions=list(range(len(chart_rows)));width=0.32
    for offset,key,label,color in [(-width/2,'p50_ms','Median','#315dde'),(width/2,'p95_ms','95th percentile','#24777c')]:
        values=[float(r[key]) for r in chart_rows]
        bars=ax.bar([p+offset for p in positions],values,width,label=label,color=color)
        ax.bar_label(bars,labels=[f'{v:.0f}' for v in values],padding=4,fontsize=10)
    ax.set_xticks(positions,['Sequential (n=20)','Three concurrent clients (n=12)'])
    ax.set_ylabel('HTTP response time (ms)');ax.set_title('Hosted prediction response times')
    ax.set_ylim(0,max(float(r['p95_ms']) for r in chart_rows)*1.28)
    ax.spines[['top','right']].set_visible(False);ax.yaxis.grid(True,alpha=.18);ax.set_axisbelow(True)
    ax.legend(frameon=False,loc='upper left')
    fig.text(.5,.02,'Full HTTP round trip and result validation. One client location; no confirmed cold-start measurement.',ha='center',fontsize=8)
    fig.tight_layout(rect=(0,.06,1,1));fig.savefig(OUT/'hosted_latency.png',dpi=240,bbox_inches='tight');plt.close(fig)
    # Refresh the manifest after adding the final publication figure.
    manifest_path=OUT/'evidence_manifest.json'
    if manifest_path.is_file():
        manifest=json.loads(manifest_path.read_text());file=OUT/'hosted_latency.png'
        manifest['files']=[r for r in manifest['files'] if r['file']!=file.name]
        manifest['files'].append({'file':file.name,'bytes':file.stat().st_size,'sha256':hashlib.sha256(file.read_bytes()).hexdigest()})
        write_json('evidence_manifest.json',manifest)
    print('Saved the hosted response-time figure from measured request results.')

Saved the hosted response-time figure from measured request results.
